# Round mosaics — live, no-FFC quick look at every round

Builds a mosaic (one per real imaging color) for EVERY round in
`round_info.csv`, from a single frame per FOV near `TARGET_Z_UM` -- no flat-
field correction, no full z-stack read, so it's light enough to run
continuously alongside a real acquisition (reading straight off the NAS
while HAL/Dave is still writing). This is a quick-look tool, not a
replacement for `analysis/02_round_scheduler.ipynb`'s production mosaics
(mid-z, optional FFC, built only once a round is 100% done) -- both can run
at the same time without conflicting; this notebook saves to
`SAMPLE_DIR/figures/`, not `analysis/mosaics/`.

**What it does**: for the round currently being imaged (auto-detected, same
logic as `imaged_fovs.ipynb`), each poll reads any newly-appeared FOV's
single frame per color, builds its thumbnail, and redraws that round's
mosaic(s) so far. Once a round finishes, its final mosaic is saved and the
notebook automatically moves on to watching the next round -- this is meant
to be started once and left running for the whole experiment, not just one
round. Already-finished rounds present when you start (e.g. if you start
this partway through the experiment) are mosaic'd once in a one-time
catch-up pass before the live loop begins.

**Usage**: run every cell once, then run the last cell and leave it running.
Interrupt the kernel to stop at any time -- whatever's been built so far is
already saved.

## 1 — Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import read_image_frames
from MERci.acquisition.configs    import find_frame_table_for_hal_config
from MERci.analysis.fov           import create_thumbnail
from MERci.analysis.round         import create_mosaic
from MERci.scheduler              import resolve_round_flip_y

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # must match what HAL is writing
NOTEBOOK_NAME = "round_mosaics"   # used to namespace this notebook's figure files

# Real stage z (um) to build every round's mosaic at -- for each round/color,
# the frame whose own z is closest to this gets used. One frame per FOV per
# color, no z-stack, no FFC -- deliberately light enough to run continuously
# during a real acquisition. Change and re-run section 3 to pick a different
# depth (e.g. to match where your tissue actually has signal).
TARGET_Z_UM = 10.0

POLL_INTERVAL_SEC = 5         # must be well under the time to acquire one FOV
MAX_RUNTIME_MIN    = 24*60*7  # safety cap -- this notebook is meant to run for a
                               # whole multi-round experiment, not just one round

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

THUMBNAILS_DIR = config.analysis_dir / "thumbnails"   # shared with 01_fov_scheduler.ipynb's own convention
THUMBNAILS_DIR.mkdir(parents=True, exist_ok=True)
# Deliberately SAMPLE_DIR/figures/, not analysis/mosaics/ -- see markdown
# above: this is a quick-look tool, kept out of the way of the production
# mosaics analysis/02_round_scheduler.ipynb builds at the same filenames.
FIGURES_DIR = SAMPLE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name  : {SAMPLE_NAME}")
print(f"FOVs (total) : {meta.n_fovs}")
print(f"Rounds       : {sorted(meta.rounds)}")
print(f"TARGET_Z_UM  : {TARGET_Z_UM}")

## 3 — Resolve each round's real colors -> nearest-z frame index

Reads each round's own frame table (via its HAL config, same resolution
`prepare_imaging` already uses) and picks, per real (non-blank) color, the
frame whose actual `z` (um) is closest to `TARGET_Z_UM`. Rounds sharing the
same HAL config (common for repeated bits rounds) resolve identically, but
each round is still looked up independently since nothing guarantees that in
general.

In [ ]:
def resolve_round_color_frames(round_id):
    # {color_nm: frame_idx} for round_id's own frame table -- the frame
    # closest to TARGET_Z_UM for every real (non-blank) color it has.
    color_frames = {}
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        frame_table_path = find_frame_table_for_hal_config(
            config.settings_dir / s.hal_config, config.metadata_dir)
        if frame_table_path is None:
            continue
        frame_table = pd.read_csv(frame_table_path)
        for color in sorted(frame_table["color"].dropna().unique()):
            candidates = frame_table[frame_table["color"].round(0) == round(color)]
            frame_idx = int((candidates["z"] - TARGET_Z_UM).abs().idxmin())
            resolved_z = float(candidates.loc[frame_idx, "z"])
            color_frames[float(color)] = frame_idx
            if abs(resolved_z - TARGET_Z_UM) > 5.0:
                print(f"  round {round_id}, color {color:.0f} nm: nearest available z is "
                      f"{resolved_z:.1f} um (requested {TARGET_Z_UM:.1f} um) -- frame {frame_idx}")
    return color_frames


ROUND_COLOR_FRAMES = {}
for round_id in sorted(meta.rounds):
    cf = resolve_round_color_frames(round_id)
    if cf:
        ROUND_COLOR_FRAMES[round_id] = cf
    print(f"Round {round_id}: colors {sorted(cf)} -> frame indices {cf}")

## 4 — Catch-up pass: mosaic every already-finished round once

Only rounds with 100% of their FOVs already imaged, and only if that round's
mosaic file(s) don't already exist yet (so re-running this notebook after an
interruption doesn't redo finished work). The round currently being imaged
(if any) is deliberately skipped here -- section 5's live loop handles it,
updating its mosaic as new FOVs actually appear rather than only once.

In [ ]:
def round_n_imaged(round_id):
    series = meta.series_for_round(round_id)
    return sum(
        1 for fov_id in sorted(meta.fovs)
        if any(s.resolve_path(fov_id, config.image_suffix).exists() for s in series)
    )


def mosaic_path(round_id, color_nm):
    return FIGURES_DIR / f"{NOTEBOOK_NAME}.round{round_id:03d}_{color_nm:.0f}nm.png"


def build_round_mosaic(round_id, color_frames, fov_ids):
    # Read one frame per fov_id per color, thumbnail it (cached to
    # THUMBNAILS_DIR, shared with 01_fov_scheduler.ipynb's own convention), and
    # save/return one mosaic per color. Returns {color_nm: canvas}.
    series  = meta.series_for_round(round_id)
    flip_y  = resolve_round_flip_y(round_id, config, meta)
    canvases = {}
    for color_nm, frame_idx in color_frames.items():
        thumbnails, positions = {}, {}
        for fov_id in fov_ids:
            existing = [s.resolve_path(fov_id, config.image_suffix) for s in series]
            existing = [p for p in existing if p.exists()]
            if not existing:
                continue
            image_path = existing[0]
            thumb_path = THUMBNAILS_DIR / f"{image_path.stem}_frame{frame_idx:03d}.png"
            if thumb_path.exists():
                from PIL import Image
                thumb = np.array(Image.open(str(thumb_path)))
            else:
                frame = read_image_frames(
                    image_path, [frame_idx],
                    frame_width=config.frame_width, frame_height=config.frame_height,
                )[0]
                thumb = create_thumbnail(frame, thumb_path, target_size=config.thumbnail_size,
                                          percentile_clip=config.thumbnail_percentile_clip)
            thumbnails[fov_id] = thumb
            positions[fov_id]  = meta.fovs[fov_id].position
        if not thumbnails:
            continue
        canvases[color_nm] = create_mosaic(
            thumbnails, positions, mosaic_path(round_id, color_nm),
            thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding, flip_y=flip_y,
        )
    return canvases


for round_id, color_frames in ROUND_COLOR_FRAMES.items():
    if round_n_imaged(round_id) < meta.n_fovs:
        continue   # not finished -- section 5's live loop will pick it up when it's the active round
    if all(mosaic_path(round_id, c).exists() for c in color_frames):
        continue   # already built by a previous run of this notebook
    print(f"Catch-up: building mosaic(s) for already-finished round {round_id}...")
    build_round_mosaic(round_id, color_frames, sorted(meta.fovs))
print("Catch-up pass done.")

## 5 — Live loop: watch the active round, advance automatically as rounds finish

Same round auto-detection as `imaged_fovs.ipynb` (prefers a round with SOME
but not ALL FOVs imaged; during a fluidics gap, points at the round after the
most recently completed one). Unlike `imaged_fovs.ipynb`, this loop keeps
running across MULTIPLE rounds for the whole experiment -- once the round
it's watching finishes, it saves that round's final mosaic, frees its
in-memory thumbnails, and moves on to whichever round is next.

Interrupt the kernel to stop -- whatever's been built so far is already
saved to `figures/`.

In [ ]:
def detect_active_round():
    best_round, best_latest, best_in_progress = None, -1.0, False
    for round_id in ROUND_COLOR_FRAMES:
        n_imaged = round_n_imaged(round_id)
        if n_imaged == 0:
            continue
        in_progress = n_imaged < meta.n_fovs
        # No real mtime scan here (unlike imaged_fovs.ipynb) -- n_imaged alone
        # is enough to rank candidates since round ids already have a real
        # chronological order (rounds are imaged strictly in round_id order).
        latest = float(round_id)
        if (in_progress, latest) > (best_in_progress, best_latest):
            best_round, best_latest, best_in_progress = round_id, latest, in_progress

    if best_round is None:
        return sorted(ROUND_COLOR_FRAMES)[0]

    if not best_in_progress:
        round_ids = sorted(ROUND_COLOR_FRAMES)
        idx = round_ids.index(best_round)
        if idx + 1 < len(round_ids):
            return round_ids[idx + 1]

    return best_round


watched_round   = None
thumbnails_by_color = {}   # {color_nm: {fov_id: thumbnail}}
imaged_fov_ids  = set()

start_time = time.time()
poll_count = 0

try:
    while True:
        if (time.time() - start_time) > MAX_RUNTIME_MIN * 60:
            print(f"Stopping: MAX_RUNTIME_MIN={MAX_RUNTIME_MIN} exceeded.")
            break

        active_round = detect_active_round()
        if active_round != watched_round:
            watched_round = active_round
            thumbnails_by_color = {c: {} for c in ROUND_COLOR_FRAMES[watched_round]}
            imaged_fov_ids = set()
            print(f"Now watching round {watched_round} "
                  f"(colors: {sorted(ROUND_COLOR_FRAMES[watched_round])}).")

        color_frames = ROUND_COLOR_FRAMES[watched_round]
        series       = meta.series_for_round(watched_round)
        flip_y       = resolve_round_flip_y(watched_round, config, meta)

        poll_count += 1
        newly_imaged = []
        for fov_id in sorted(meta.fovs):
            if fov_id in imaged_fov_ids:
                continue
            existing = [s.resolve_path(fov_id, config.image_suffix) for s in series]
            existing = [p for p in existing if p.exists()]
            if existing:
                imaged_fov_ids.add(fov_id)
                newly_imaged.append((fov_id, existing[0]))

        for fov_id, image_path in newly_imaged:
            for color_nm, frame_idx in color_frames.items():
                frame = read_image_frames(
                    image_path, [frame_idx],
                    frame_width=config.frame_width, frame_height=config.frame_height,
                )[0]
                thumb_path = THUMBNAILS_DIR / f"{image_path.stem}_frame{frame_idx:03d}.png"
                thumb = create_thumbnail(frame, thumb_path, target_size=config.thumbnail_size,
                                          percentile_clip=config.thumbnail_percentile_clip)
                thumbnails_by_color[color_nm][fov_id] = thumb

        colors = sorted(color_frames)
        fig, axes = plt.subplots(1, len(colors), figsize=(6 * len(colors), 6), squeeze=False)
        for ax, color_nm in zip(axes[0], colors):
            thumbs = thumbnails_by_color[color_nm]
            if thumbs:
                positions = {f: meta.fovs[f].position for f in thumbs}
                canvas = create_mosaic(
                    thumbs, positions, mosaic_path(watched_round, color_nm),
                    thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding, flip_y=flip_y,
                )
                ax.imshow(canvas, cmap="gray")
            ax.set_title(f"round {watched_round} — {color_nm:.0f} nm")
            ax.axis("off")
        fig.tight_layout()

        clear_output(wait=True)
        display(fig)
        plt.close(fig)
        elapsed = time.time() - start_time
        print(f"poll #{poll_count} | elapsed {elapsed / 60:.1f} min | round {watched_round} | "
              f"imaged {len(imaged_fov_ids)}/{meta.n_fovs} | next check in {POLL_INTERVAL_SEC}s")
        # Once this round is fully imaged, the top-of-loop `active_round !=
        # watched_round` check will detect the switch to the next round on
        # the NEXT iteration and reset thumbnails_by_color there -- UNLESS
        # this is already the last round, which has no next round to switch
        # to (detect_active_round's own fallback just keeps returning it) --
        # stop the whole loop in that case rather than polling a finished
        # experiment forever.
        if watched_round == max(ROUND_COLOR_FRAMES) and len(imaged_fov_ids) >= meta.n_fovs:
            print("All rounds finished.")
            break

        time.sleep(POLL_INTERVAL_SEC)
except KeyboardInterrupt:
    print(f"Stopped by user while watching round {watched_round} "
          f"({len(imaged_fov_ids)}/{meta.n_fovs} FOVs imaged).")